# DenseNet

In [15]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd

In [2]:
from tensorflow.keras import layers, models
from tensorflow.keras import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import concatenate
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Layer, Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPool2D, AveragePooling2D
from tensorflow.keras.layers import BatchNormalization, Activation, GlobalAvgPool2D

In [70]:
# composite function 구현
def Composite_func(x, filters = 32, kernel = 1, strides = 1):
    x = BatchNormalization()(x)
    x = Activation(keras.activations.relu)(x)
    x = Conv2D(filters, kernel, strides = strides, padding = 'same')(x)
    
    return x

# Dense_Block 구현 : Bottle neck 형태로 진행됨
def Dense_Block(x, repeat):
    array = [x]
    for i in range(repeat):
        filters = 32 # k값
        y = Composite_func(x, 4*filters, 1, 1) # 1x1 Conv
        y = Composite_func(y, filters, 3, 1) # 3x3 Conv
        array.append(y)
        x = concatenate(array) # concat 연산으로 feature map 더해줌
        
    return x

# transition layer 구현
def transition(x):
    x = BatchNormalization()(x)
    x = Conv2D(12, (1,1), strides = 1, padding = 'same')(x)
    x = AveragePooling2D(2, strides = 2, padding = 'same')(x)
    
    return x

In [73]:
input_x = Input(shape = (224, 224, 3))

# 초기 layer의 k는 2k로 32*2 = 64
x = Conv2D(64, 7, strides = 2, padding = 'same', use_bias = False)(input_x)
x = MaxPool2D(pool_size = 3, strides = 2, padding = 'same')(x)

# 각 Dense Block, transition을 6, 12, 24, 16번 반복
for repeat in [6, 12, 24, 16]:
    d = Dense_Block(x, repeat)
    x = transition(d)

# 전체 feature map을 GlobalAvgPool 수행
x = GlobalAvgPool2D()(d) # 마지막 Dense Block은 transition 수행 안함
output = Dense(1000, activation = 'softmax')(x)

model = Model(input_x, output)
model.summary()

Model: "model_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_34 (InputLayer)           [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
conv2d_624 (Conv2D)             (None, 112, 112, 64) 9408        input_34[0][0]                   
__________________________________________________________________________________________________
max_pooling2d_51 (MaxPooling2D) (None, 56, 56, 64)   0           conv2d_624[0][0]                 
__________________________________________________________________________________________________
batch_normalization_556 (BatchN (None, 56, 56, 64)   256         max_pooling2d_51[0][0]           
____________________________________________________________________________________________